In [ ]:
import os, sys, json, subprocess, shutil
from pathlib import Path
repo = Path(os.environ['SOUP_REPO'])
results = Path(os.environ['SOUP_RESULTS'])
print('Editable challenge repo:', repo)
print('Public data:', list((repo / 'data').glob('*.jsonl')))
print('Write your solution files under', repo / 'solution')
# Add notebook cells here to replace solution/__init__.py and your kernels.
# The prepared starter is retained when you run this template unchanged.


In [ ]:
# Offline candidate install cell. Run this cell once before the server's
# separate-process solution invocation. Requires the inspected starter checkout
# at Path(os.environ["SOUP_REPO"]), including unchanged solution/swiglu.py and
# solution/test_kernels.py. No network, dependency installation, GPU imports,
# training, tests, or apply() invocation occur in this cell.
#
# All 28 solution Python sources are embedded below. The two protected starter
# files are verified only; the other 26 are installed when missing or different.
# Unexpected files are rejected before writes. Existing __pycache__ directories
# are left untouched, as in the official whole-solution hash.


def _install_soup_candidate():
    import ast
    import base64
    import hashlib
    import io
    import json
    import os
    from pathlib import Path, PurePosixPath
    import stat
    import tempfile
    import zipfile

    expected_solution_sha256 = 'f749d20c06ad60390bace705f443067f455e0b169b5441047c040ebb3f9bd885'
    expected_archive_sha256 = '237f15c4e1df17fce2de1d1cdad869c182ae7945d8851fc04bb45bd5aa4b8561'
    protected_names = ('swiglu.py', 'test_kernels.py')
    expected_settings = {'CHUNK_TOKENS': 256,
     'COMPACT_MLP_ACTIVATIONS': True,
     'FROZEN_STORAGE': True,
     'FUSED_LORA_EPILOGUE': False,
     'FUSED_NF4': True,
     'FUSED_ROTARY': False,
     'HEAD_NUMERICS': 'native',
     'HEAD_PROJECTION': 'torch',
     'SPARSE_BACKWARD': True}
    manifest = {'__init__.py': {'bytes': 1873,
                     'sha256': 'd6556e8e878ffc7cf63680459cde21aee05cef071953b9ffadce21ba15c63435'},
     'activation_storage.py': {'bytes': 7404,
                               'sha256': '95f66325e0361ced6e0c56ad06a027c66ac8d371969dc44a5840f746dfd21ca8'},
     'frozen_storage.py': {'bytes': 8584,
                           'sha256': 'c27dc833f9b7ceb15b3f7e96169c5a6572060abead71c093aac8625636aa0fe0'},
     'head_fused_t4.py': {'bytes': 26490,
                          'sha256': 'ac75f93ce5b46eeb564afd8c74409cbbeda36804c0399d639c159bf8779b14ce'},
     'head_matmul.py': {'bytes': 4134,
                        'sha256': 'c834bc001e3c6f9449ef4f5cd60d34aa961262e27c81e98e9a47186af5ec7e39'},
     'head_matmul_t4.py': {'bytes': 9051,
                           'sha256': 'b29ed64e0d1537f6682d2b75d5a979c8269030a306aa8f1dee263bb9e3be3134'},
     'integration.py': {'bytes': 12622,
                        'sha256': '3da67233c3917e5efc5e1811f096664a95e8500007533e5bb1f921c2bcbb7a2d'},
     'lora_epilogue.py': {'bytes': 9912,
                          'sha256': '124f3ff31c7f1328bfa2a0e7760fd6bd7b48d9daa654d1addef154760767964e'},
     'metric_transfer.py': {'bytes': 6052,
                            'sha256': '2b30a1c1ec61eca8841a9f2f970588f65ce2847b6b178ebd7d950995dba5a2d9'},
     'nf4.py': {'bytes': 10430,
                'sha256': '9e07a73d45f7b7119fc1f5d1d3422351e5d34e2fbd9612f7b90f56bc17ad697a'},
     'rmsnorm.py': {'bytes': 5869,
                    'sha256': 'aa55a21fd921d2f5478444b03a32694a329bb10bbbe4f3bf46e4e78f2804a93c'},
     'rotary.py': {'bytes': 13338,
                   'sha256': '8e8ef322d70eba6ac898a040a14d74b9dc8e276b3f93bd033414eeaf927b5429'},
     'settings.py': {'bytes': 1489,
                     'sha256': '17261889d0b049ddc41a2607e7106dc62e466069466344bcab885be84f9494de'},
     'streamed_head.py': {'bytes': 27782,
                          'sha256': '65ad6cada653b0d03d70375570b132c69f47a52ee8ff65ef0cb11042a63c9f19'},
     'swiglu.py': {'bytes': 3185,
                   'sha256': 'e665910faeca6091c958ed7a191604484a44216f51d6fa069f5f48220e7472e6'},
     'test_activation_storage.py': {'bytes': 14607,
                                    'sha256': '4a4b8b9819c26745fb10ef8228119981fec75012b140844c43552f0ac42291cf'},
     'test_frozen_storage.py': {'bytes': 11574,
                                'sha256': '87c98888cdb260d6e0e3e2c639ab5e9107a26da1708b8b005a2a4aa1d61ef07d'},
     'test_head_fused_t4.py': {'bytes': 24883,
                               'sha256': '335d684aa6fdc063d478265209e6ee05fcdf32fa7babe8d3e6c24644679d512d'},
     'test_head_matmul.py': {'bytes': 2743,
                             'sha256': '1ae409f55282fbd96d3c78845ab22cded87bf453149fbecbbfe7e3f62ec4c9bd'},
     'test_head_matmul_t4.py': {'bytes': 7100,
                                'sha256': '75931a45b9623e2ceb458904166218d31ffc68ccb08f129da5691cbdc961467d'},
     'test_integration.py': {'bytes': 30248,
                             'sha256': 'e8b5c5571d8baa987e38affa925defbf10de29ef025771fa20c74d41101e6aea'},
     'test_kernels.py': {'bytes': 1300,
                         'sha256': 'a45b842a4f22c1aefe404baeff1ac26511253452ecb0f633c985fbc9df8b252a'},
     'test_lora_epilogue.py': {'bytes': 15228,
                               'sha256': 'c7394c871558fe12ab684aca5e5872ecd18763467e2ace460ff9016c7efd6421'},
     'test_nf4.py': {'bytes': 10862,
                     'sha256': '2ee2f46a3011ab76c61675bef3b3e1ce40e0964d1737c92c36b71e8fd05c9a4f'},
     'test_rmsnorm.py': {'bytes': 3634,
                         'sha256': 'a0ef2638f9d7704c5da4154dcf9eac2b87069952fd3ff743da63ea84ddbdfded'},
     'test_rotary.py': {'bytes': 18180,
                        'sha256': 'a8659bbb8455c6f20868672a7a41e4c96c67b5a066cd443addd0067acf4faf37'},
     'test_streamed_head.py': {'bytes': 29925,
                               'sha256': '5ebbca158118591e39fca8b70ecf9f2485b287f07715573ab1c271acf21775d4'},
     'test_training_parity.py': {'bytes': 3303,
                                 'sha256': '412b9ebcd7d1d95c27b0e388a43f9b71820c3959ed5caa04345c09ba9fa945f1'}}
    payload_b64 = '...'

    # Decode in memory; never extract an archive directly onto the filesystem.
    archive_bytes = base64.b64decode(payload_b64, validate=True)
    if hashlib.sha256(archive_bytes).hexdigest() != expected_archive_sha256:
        raise RuntimeError("Embedded candidate archive checksum mismatch")
    payload = {}
    with zipfile.ZipFile(io.BytesIO(archive_bytes), "r") as archive:
        members = archive.infolist()
        names = [member.filename for member in members]
        if len(names) != len(set(names)) or set(names) != set(manifest):
            raise RuntimeError("Embedded archive has missing, duplicate, or unexpected entries")
        for member in members:
            name = member.filename
            path = PurePosixPath(name)
            if (path.is_absolute() or len(path.parts) != 1 or path.name != name
                    or "\\" in name or not name.endswith(".py")
                    or member.is_dir() or stat.S_ISLNK(member.external_attr >> 16)):
                raise RuntimeError("Unsafe embedded archive path: " + repr(name))
            expected = manifest[name]
            if member.file_size != expected["bytes"]:
                raise RuntimeError("Embedded source size mismatch: " + name)
            content = archive.read(member)
            if (len(content) != expected["bytes"]
                    or hashlib.sha256(content).hexdigest() != expected["sha256"]):
                raise RuntimeError("Embedded source checksum mismatch: " + name)
            compile(content, "solution/" + name, "exec")  # Parse only; execute nothing.
            payload[name] = content

    # Check settings and the no-argument entry point without importing solution.
    settings = {}
    for node in ast.parse(payload["settings.py"]).body:
        if isinstance(node, ast.Assign):
            for lhs in node.targets:
                if isinstance(lhs, ast.Name):
                    settings[lhs.id] = ast.literal_eval(node.value)
    if settings != expected_settings:
        raise RuntimeError("Embedded candidate settings differ from the fixed candidate")
    entrypoints = [node for node in ast.parse(payload["__init__.py"]).body
                   if isinstance(node, ast.FunctionDef) and node.name == "apply"]
    if len(entrypoints) != 1:
        raise RuntimeError("Embedded candidate must define exactly one apply()")
    args = entrypoints[0].args
    if args.posonlyargs or args.args or args.kwonlyargs or args.vararg or args.kwarg:
        raise RuntimeError("Embedded candidate apply() must take no arguments")

    repo_value = os.environ.get("SOUP_REPO")
    if repo_value is None or not repo_value.strip():
        raise RuntimeError("Set SOUP_REPO to the existing official starter checkout first")
    repo = Path(repo_value).resolve(strict=True)
    if not repo.is_dir():
        raise RuntimeError("SOUP_REPO must name an existing directory")
    solution = repo / "solution"
    if solution.is_symlink() or not solution.is_dir():
        raise RuntimeError("SOUP_REPO/solution must be an existing real directory")

    # Preflight every destination before any write. Never delete unknown files.
    unexpected = []
    for path in sorted(solution.rglob("*")):
        relative = path.relative_to(solution)
        name = str(relative)
        if path.is_symlink():
            raise RuntimeError("Refusing a symlink in solution/: " + name)
        if "__pycache__" in relative.parts:
            continue  # Official hash excludes these; do not modify or remove them.
        if not path.is_file() or name not in manifest:
            unexpected.append(name)
    if unexpected:
        raise RuntimeError("Unexpected solution/ entries; inspect them before retrying: "
                           + ", ".join(unexpected))
    for name in protected_names:
        path = solution / name
        if not path.is_file() or path.read_bytes() != payload[name]:
            raise RuntimeError("Protected starter source missing or different: solution/"
                               + name + "; no files were written")

    updated = []
    unchanged = []
    for name, content in sorted(payload.items()):
        destination = solution / name
        if name in protected_names:
            continue  # Verify-only, even when the bytes already match.
        if destination.is_file() and destination.read_bytes() == content:
            unchanged.append(name)
            continue
        # Atomic replacement avoids partially written source files and does not
        # follow a destination hardlink. Temporary files stay inside solution/.
        temporary = None
        try:
            with tempfile.NamedTemporaryFile(mode="wb", dir=solution,
                                             prefix="._soup_install_", suffix=".tmp",
                                             delete=False) as handle:
                temporary = Path(handle.name)
                handle.write(content)
                handle.flush()
                os.fchmod(handle.fileno(), 0o644)
            os.replace(temporary, destination)
            temporary = None
            updated.append(name)
        finally:
            if temporary is not None:
                try:
                    temporary.unlink()  # Only the temporary file created above.
                except FileNotFoundError:
                    pass

    # Same algorithm as unchanged harness/check_rules.py:solution_sha256.
    # It hashes relative path bytes followed by file bytes in sorted path order.
    digest = hashlib.sha256()
    for path in sorted(Path(solution).rglob("*")):
        if path.is_file() and "__pycache__" not in path.parts:
            digest.update(str(path.relative_to(solution)).encode())
            digest.update(path.read_bytes())
    observed_solution_sha256 = digest.hexdigest()
    if observed_solution_sha256 != expected_solution_sha256:
        raise RuntimeError("Installed whole-solution hash mismatch: "
                           + observed_solution_sha256)
    print(json.dumps({
        "status": "installed_and_verified",
        "solution_sha256": observed_solution_sha256,
        "source_files": len(manifest),
        "updated_files": len(updated),
        "already_matching_files": len(unchanged),
        "protected_files_verified": list(protected_names),
        "settings": settings,
        "apply_called": False,
    }, indent=2, sort_keys=True))


_install_soup_candidate()


In [ ]:
# Experiment: cache exact decoded FP16 NF4 base weights across the measured run.
# This uses the storage-reduced f749d20 base so the full 28-layer cache fits.
# It patches only our own solution sources after the SHA-verified installer.
from pathlib import Path
import hashlib

nf4_path = repo / "solution" / "nf4.py"
init_path = repo / "solution" / "__init__.py"
nf4_src = nf4_path.read_text(encoding="utf-8")
init_src = init_path.read_text(encoding="utf-8")
assert hashlib.sha256(nf4_src.encode()).hexdigest() == "9e07a73d45f7b7119fc1f5d1d3422351e5d34e2fbd9612f7b90f56bc17ad697a"
assert hashlib.sha256(init_src.encode()).hexdigest() == "d6556e8e878ffc7cf63680459cde21aee05cef071953b9ffadce21ba15c63435"

nf4_src += r'''

_NF4_WEIGHT_CACHE = {}
_NF4_WEIGHT_CACHE_BYTES = 0
_NF4_WEIGHT_CACHE_LIMIT = 3 * 1024**3


def _cache_tensor_signature(tensor):
    if tensor is None:
        return None
    storage = tensor.untyped_storage()
    return (
        storage.data_ptr(), storage.nbytes(), tensor.storage_offset(),
        tuple(tensor.shape), tuple(tensor.stride()), tensor.dtype,
        tensor.device, tensor._version,
    )


def _cache_components_key(components):
    return (
        _cache_tensor_signature(components["packed"]),
        tuple(components["shape"]), components["dtype"], components["blocksize"],
        _cache_tensor_signature(components["scales"]),
        _cache_tensor_signature(components.get("nested_scales")),
        _cache_tensor_signature(components.get("nested_code")),
        _cache_tensor_signature(components.get("offset")),
        components.get("nested_blocksize", 256),
    )


def _cache_stream_supported(device):
    if torch.cuda.is_current_stream_capturing():
        return False
    current = torch.cuda.current_stream(device)
    default = torch.cuda.default_stream(device)
    return current.cuda_stream == default.cuda_stream


def _cache_decoded_weight(components):
    global _NF4_WEIGHT_CACHE_BYTES
    if components.get("out") is not None or components["dtype"] != torch.float16:
        return None
    packed = components["packed"]
    if (not packed.is_cuda or packed.requires_grad
            or not _cache_stream_supported(packed.device)):
        return None

    key = _cache_components_key(components)
    entry = _NF4_WEIGHT_CACHE.get(key)
    if entry is not None:
        return entry[0]

    decoded = _execute(**components)
    size = decoded.numel() * decoded.element_size()
    if _NF4_WEIGHT_CACHE_BYTES + size > _NF4_WEIGHT_CACHE_LIMIT:
        return decoded

    # Hold the immutable packed/metadata tensors as ownership guards so a later
    # allocator reuse cannot alias an entry with the same raw addresses.
    refs = (
        components["packed"], components["scales"],
        components.get("nested_scales"), components.get("nested_code"),
        components.get("offset"),
    )
    _NF4_WEIGHT_CACHE[key] = (decoded, refs)
    _NF4_WEIGHT_CACHE_BYTES += size
    return decoded


def patch_nf4_weight_cache():
    """Reuse exact decoded frozen NF4 weights at both reviewed bnb boundaries."""
    import bitsandbytes.functional as functional
    from bitsandbytes.backends.cuda import ops

    if getattr(functional.dequantize_4bit, "_soup_full_nf4_cache", False):
        return True
    if not getattr(functional.dequantize_4bit, "_soup_candidate_nf4", False):
        return False

    original_dequantize = functional.dequantize_4bit
    original_fallback = ops._dequant_linear_fallback

    @wraps(original_dequantize)
    def cached_dequantize_4bit(A, quant_state=None, absmax=None, out=None,
                               blocksize=None, quant_type="fp4"):
        components = _state_components(A, quant_state, out, cuda_only=True)
        if components is None:
            return original_dequantize(
                A, quant_state, absmax, out, blocksize, quant_type
            )
        decoded = _cache_decoded_weight(components)
        if decoded is None:
            return original_dequantize(
                A, quant_state, absmax, out, blocksize, quant_type
            )
        return decoded.t() if A.shape[0] == 1 else decoded

    @wraps(original_fallback)
    def cached_dequant_linear_fallback(
        A, B, shapeB, absmax, blocksize, quant_type,
        bias=None, absmax_8bit=None, absmax_code=None, absmax_offset=None
    ):
        nested = absmax_8bit is not None
        components = dict(
            packed=B, shape=shapeB, dtype=A.dtype,
            scales=absmax_8bit if nested else absmax,
            blocksize=blocksize,
            nested_scales=absmax if nested else None,
            nested_code=absmax_code,
            offset=absmax_offset,
            nested_blocksize=256,
        )
        if (quant_type != "nf4" or A.device != B.device
                or not _components_supported(**components, cuda_only=True)):
            return original_fallback(
                A, B, shapeB, absmax, blocksize, quant_type, bias,
                absmax_8bit, absmax_code, absmax_offset
            )
        decoded = _cache_decoded_weight(components)
        if decoded is None:
            return original_fallback(
                A, B, shapeB, absmax, blocksize, quant_type, bias,
                absmax_8bit, absmax_code, absmax_offset
            )
        return torch.nn.functional.linear(A, decoded, bias)

    cached_dequantize_4bit._soup_candidate_nf4 = True
    cached_dequantize_4bit._soup_full_nf4_cache = True
    cached_dequant_linear_fallback._soup_candidate_nf4 = True
    cached_dequant_linear_fallback._soup_full_nf4_cache = True
    functional.dequantize_4bit = cached_dequantize_4bit
    ops._dequant_linear_fallback = cached_dequant_linear_fallback
    return True
'''

anchor = '''        from .nf4 import patch_bitsandbytes_nf4
        patch_bitsandbytes_nf4()
'''
replacement = '''        from .nf4 import patch_bitsandbytes_nf4, patch_nf4_weight_cache
        patch_bitsandbytes_nf4()
        patch_nf4_weight_cache()
'''
assert init_src.count(anchor) == 1
init_src = init_src.replace(anchor, replacement, 1)

compile(nf4_src, str(nf4_path), "exec")
compile(init_src, str(init_path), "exec")
nf4_path.write_text(nf4_src, encoding="utf-8")
init_path.write_text(init_src, encoding="utf-8")
print("full_nf4_cache_patch=installed")
print("nf4_sha256=", hashlib.sha256(nf4_src.encode()).hexdigest())
print("init_sha256=", hashlib.sha256(init_src.encode()).hexdigest())


In [ ]:
# Fixed experiment: compact frozen MLP and fuse its residual scatter
# into a single original Triton kernel. No trainable parameter or task change.
from pathlib import Path
import hashlib, json
init_path = repo / "solution" / "__init__.py"
init_src = init_path.read_text(encoding="utf-8")
assert init_path.is_file()
assert init_src.count("patch_nf4_weight_cache()") == 1
assert init_src.count("patch_mlp_padding_compaction()") == 0
padding_src = "\"\"\"Original fused padded MLP residual epilogue for Qwen2.5 T4.\n\nCompacts *only* the padding-masked frozen MLP branch within each unchanged\nbatch.  Q/V LoRA dropout, attention, position encodings, residual paths,\nbatch order, optimizer, and Soup metrics remain untouched.  The fused Triton\nkernel replaces a zeros_like, indexed scatter, and dense residual add.\n\"\"\"\n\nimport contextvars\nimport inspect\nimport torch\nimport triton\nimport triton.language as tl\n\n_ACTIVE_ROWS = contextvars.ContextVar(\"_soup_compact_mlp_rows\", default=None)\n\n\n@triton.jit\ndef soup_mlp_residual_scatter(RESIDUAL, COMPACT, ROW_MASK, PREFIX, OUTPUT,\n                               WIDTH: tl.constexpr, BLOCK: tl.constexpr):\n    row = tl.program_id(0)\n    col = tl.arange(0, BLOCK)\n    valid = col < WIDTH\n    keep = tl.load(ROW_MASK + row)\n    idx = tl.maximum(tl.load(PREFIX + row), 0)\n    residual = tl.load(RESIDUAL + row * WIDTH + col, valid, 0).to(tl.float32)\n    branch = tl.load(COMPACT + idx * WIDTH + col, keep & valid, other=0.0).to(tl.float32)\n    tl.store(OUTPUT + row * WIDTH + col, residual + branch, valid)\n\n\nclass _FusedResidual(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, residual, compact, active, row_mask, prefix):\n        nrows, width = residual.shape\n        output = torch.empty_like(residual)\n        soup_mlp_residual_scatter[(nrows,)](\n            residual, compact, row_mask, prefix, output,\n            WIDTH=width, BLOCK=triton.next_power_of_2(width),\n            num_warps=4, enable_fp_fusion=False,\n        )\n        ctx.save_for_backward(active)\n        ctx.set_materialize_grads(False)\n        return output\n\n    @staticmethod\n    def backward(ctx, grad_output):\n        if grad_output is None:\n            return None, None, None, None, None\n        (active,) = ctx.saved_tensors\n        # d(residual + compact) / dresidual = 1, and dcompact selects\n        # exactly the indexed, unpadded rows. Gradient accumulation is FP32.\n        return grad_output, grad_output.index_select(0, active), None, None, None\n\n\ndef patch_mlp_padding_compaction():\n    from transformers.models.qwen2 import modeling_qwen2\n\n    model_cls = modeling_qwen2.Qwen2Model\n    decoder_cls = modeling_qwen2.Qwen2DecoderLayer\n    mlp_cls = modeling_qwen2.Qwen2MLP\n\n    if getattr(decoder_cls.forward, \"_soup_fused_padded_residual\", False):\n        return True\n\n    original_model = model_cls.forward\n    original_decoder = decoder_cls.forward\n    original_mlp = mlp_cls.forward\n    model_signature = inspect.signature(original_model)\n\n    def model_forward(self, *args, **kwargs):\n        try:\n            bound = model_signature.bind_partial(self, *args, **kwargs)\n            mask = bound.arguments.get(\"attention_mask\")\n        except TypeError:\n            mask = kwargs.get(\"attention_mask\")\n        plan = None\n        if (isinstance(mask, torch.Tensor) and mask.is_cuda and mask.ndim == 2\n                and mask.numel() > 0):\n            flat_mask = (mask.reshape(-1) != 0).contiguous()\n            active = torch.nonzero(flat_mask, as_tuple=False).flatten()\n            if 0 < active.numel() < flat_mask.numel():\n                prefix = torch.cumsum(flat_mask.to(torch.int32), dim=0) - 1\n                plan = (active, tuple(mask.shape), flat_mask, prefix)\n        token = _ACTIVE_ROWS.set(plan)\n        try:\n            return original_model(self, *args, **kwargs)\n        finally:\n            _ACTIVE_ROWS.reset(token)\n\n    def decoder_forward(\n        self, hidden_states, attention_mask=None, position_ids=None,\n        past_key_values=None, use_cache=False, position_embeddings=None, **kwargs,\n    ):\n        plan = _ACTIVE_ROWS.get()\n        if (plan is None or not isinstance(hidden_states, torch.Tensor)\n                or hidden_states.ndim != 3 or not hidden_states.is_cuda\n                or hidden_states.dtype != torch.float32 or not hidden_states.is_contiguous()):\n            return original_decoder(\n                self, hidden_states, attention_mask=attention_mask,\n                position_ids=position_ids, past_key_values=past_key_values,\n                use_cache=use_cache, position_embeddings=position_embeddings, **kwargs,\n            )\n        active, shape, flat_mask, prefix = plan\n        if (tuple(hidden_states.shape[:2]) != shape or active.device != hidden_states.device\n                or not self.training or self.mlp.__class__ is not mlp_cls):\n            return original_decoder(\n                self, hidden_states, attention_mask=attention_mask,\n                position_ids=position_ids, past_key_values=past_key_values,\n                use_cache=use_cache, position_embeddings=position_embeddings, **kwargs,\n            )\n\n        # The unchanged Qwen2DecoderLayer attention and first residual.\n        residual = hidden_states\n        q = self.input_layernorm(hidden_states)\n        attended, _ = self.self_attn(\n            hidden_states=q, attention_mask=attention_mask,\n            position_ids=position_ids, past_key_values=past_key_values,\n            use_cache=use_cache, position_embeddings=position_embeddings, **kwargs,\n        )\n        residual = residual + attended\n        # RMSNorm is row-local. Gather valid rows first to avoid normalizing\n        # padded positions that the MLP path already proves unobservable.\n        # The existing original RMSNorm kernel still computes the same FP32\n        # arithmetic for each retained row and its exact gradient.\n        flat_residual = residual.reshape(-1, residual.shape[-1])\n        compact_input = flat_residual.index_select(0, active)\n        compact_normed = self.post_attention_layernorm(compact_input)\n        compact = original_mlp(self.mlp, compact_normed)\n        if (compact.ndim != 2 or compact.shape[0] != active.numel()\n                or compact.shape[1] != residual.shape[-1]\n                or compact.dtype != torch.float32 or not compact.is_contiguous()\n                or not residual.is_contiguous()):\n            out = torch.zeros_like(flat_residual)\n            out.index_copy_(0, active, compact.reshape(active.numel(), -1))\n            return residual + out.reshape_as(residual)\n\n        result = _FusedResidual.apply(\n            residual.reshape(-1, residual.shape[-1]),\n            compact, active, flat_mask, prefix,\n        )\n        return result.view_as(residual)\n\n    model_forward._soup_fused_padded_residual = True\n    decoder_forward._soup_fused_padded_residual = True\n    model_cls.forward = model_forward\n    decoder_cls.forward = decoder_forward\n    return True\n"
padding_path = repo / "solution" / "mlp_padding.py"
compile(padding_src, str(padding_path), "exec")
padding_path.write_text(padding_src, encoding="utf-8")
anchor = """    if COMPACT_MLP_ACTIVATIONS:
        from .activation_storage import patch_compact_mlp_activations
        patch_compact_mlp_activations()
"""
replacement = anchor + """    from .mlp_padding import patch_mlp_padding_compaction
    patch_mlp_padding_compaction()
"""
assert init_src.count(anchor) == 1
init_src = init_src.replace(anchor, replacement, 1)
compile(init_src, str(init_path), "exec")
init_path.write_text(init_src, encoding="utf-8")
digest = hashlib.sha256()
for path in sorted((repo / "solution").rglob("*")):
    if path.is_file() and "__pycache__" not in path.parts:
        digest.update(str(path.relative_to(repo / "solution")).encode())
        digest.update(path.read_bytes())
manifest = {
    "experiment": "full_nf4_cache_mlp_validrow_norm_fused_residual",
    "reference_candidate": "chatgpt-fullcache-mlppad-20261031-v1",
    "solution_sha256": digest.hexdigest(),
    "mlp_padding_source_sha256": hashlib.sha256(padding_src.encode()).hexdigest(),
    "original_useful_kernel": "soup_mlp_residual_scatter",
    "safe_original_training_harness": True,
}
(results / "mlp_residual_source.json").write_text(json.dumps(manifest,indent=2)+"\n")
print(manifest)


In [ ]:
# Assert the two preceding overlays remain active and the native head starts unchanged.
from pathlib import Path
import os, hashlib
_solution_root = Path(os.environ["SOUP_REPO"]) / "solution"
assert (_solution_root / "mlp_padding.py").exists(), "MLP compaction patch missing"
_head_path = _solution_root / "streamed_head.py"
assert hashlib.sha256(_head_path.read_bytes()).hexdigest() == "65ad6cada653b0d03d70375570b132c69f47a52ee8ff65ef0cb11042a63c9f19", "Unexpected head source; refuse composite"
assert "patch_nf4_weight_cache()" in (_solution_root / "__init__.py").read_text()
assert "patch_mlp_padding_compaction()" in (_solution_root / "__init__.py").read_text()
print("base=storage-f749d20, overlays=full-NF4-cache+MLP-pad, adding exact native-logits reuse")

# Combined storage + retained-logits experiment.
# Base solution hash: f749d20c06ad60390bace705f443067f455e0b169b5441047c040ebb3f9bd885.
# streamed_head.py is byte-identical to the validated dab4c473 head, so the
# exact-source guard below remains valid.
# Experiment C0: retain the forward's original FP16 logits for backward
# rather than recompute the large frozen vocabulary projection. This patch is
# applied only to our own solution/streamed_head.py after the verified installer.
from pathlib import Path
import hashlib
head_src = repo / "solution" / "streamed_head.py"
settings_src = repo / "solution" / "settings.py"
original = head_src.read_text(encoding="utf-8")
assert settings_src.read_text(encoding="utf-8").count("CHUNK_TOKENS = 256") == 1
start = original.index("class _NativeStreamedLinearCrossEntropy(")
end = original.index("\nclass _StreamedLinearCrossEntropy(", start)
before, body, after = original[:start], original[start:end], original[end:]
edits = [
    ("        predictions = torch.empty(token_count, device=hidden.device, dtype=torch.long)\n",
     "        predictions = torch.empty(token_count, device=hidden.device, dtype=torch.long)\n"
     "        # Retain the original FP16 projected scores for exact backward replay.\n"
     "        saved_logits = torch.empty((token_count, weight_half.shape[0]),\n"
     "                                   device=hidden.device, dtype=torch.float16)\n"),
    ("                scores = torch.mm(flat_hidden[start:end], weight_half.t())\n",
     "                scores = saved_logits[start:end]\n"
     "                torch.mm(flat_hidden[start:end], weight_half.t(), out=scores)\n"),
    ("        ctx.save_for_backward(flat_hidden, weight_half, targets, denominator, unsafe_ignored_rows)\n",
     "        ctx.save_for_backward(flat_hidden, weight_half, targets, denominator,\n"
     "                              unsafe_ignored_rows, saved_logits)\n"),
    ("        flat_hidden, weight_half, targets, denominator, unsafe_ignored_rows = ctx.saved_tensors\n",
     "        (flat_hidden, weight_half, targets, denominator,\n"
     "         unsafe_ignored_rows, saved_logits) = ctx.saved_tensors\n"),
    ("                scores = torch.mm(work_hidden[start:end], weight_half.t())\n",
     "                if selected_rows is None:\n"
     "                    scores = saved_logits[start:end]\n"
     "                else:\n"
     "                    scores = saved_logits.index_select(0, selected_rows[start:end])\n"),
]
for old, new in edits:
    assert body.count(old) == 1, "Unexpected native head source"
    body = body.replace(old, new, 1)
modified = before + body + after
compile(modified, str(head_src), "exec")
head_src.write_text(modified, encoding="utf-8")
assert head_src.read_text(encoding="utf-8") == modified
print("C0 forward-logits reuse installed; SHA256:", hashlib.sha256(modified.encode()).hexdigest())


In [ ]:
# Run only the solution. The server reuses its saved matching public baseline.
run = Path('/kaggle/working/training-run')
subprocess.run([sys.executable, str(repo / 'harness/run_task.py'), '--out', str(run), '--timeout', '900', '--only', 'solution'], cwd=repo, check=True)


In [ ]:
# Export small reports and logs. Large adapter/checkpoint files remain in this job's VM workspace.
for path in run.rglob('*'):
    if path.is_file() and path.suffix in ('.json', '.jsonl', '.log', '.yaml') and path.stat().st_size < 4 * 1024**2:
        target = results / path.relative_to(run)
        target.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(path, target)
print((run / 'result.json').read_text())
# This is a public-data screen. Notebook output does not establish official admission or score.
